# Redis

Redis is an in-memory store commonly used for caches, queues, locks, and short-lived state.

In [1]:
# These commands are the building blocks for cache and queue adapters.
redis_commands = [
    "SET user:1 Ada EX 300",
    "GET user:1",
    "DEL user:1",
    "LPUSH jobs job-123",
    "BRPOP jobs 5",
]

print(*redis_commands, sep="\n")

SET user:1 Ada EX 300
GET user:1
DEL user:1
LPUSH jobs job-123
BRPOP jobs 5


Use clear key names and expiration times; Redis memory is finite.

## Polished version

This is the production cache adapter. A small fake Redis client keeps the demo runnable without a server; production injects `redis.asyncio.Redis` through the same interface.

In [ ]:
import json
from typing import Optional, Protocol


# Depend on the small Redis behavior this adapter needs, not the whole library.
class RedisClient(Protocol):
    async def get(self, key: str) -> Optional[str]: ...
    async def set(self, key: str, value: str, *, ex: int) -> object: ...
    async def delete(self, key: str) -> int: ...
    async def aclose(self) -> None: ...


class Cache(Protocol):
    async def get(self, key: str) -> Optional[dict]: ...
    async def set(self, key: str, value: dict, ttl_seconds: int) -> None: ...
    async def delete(self, key: str) -> None: ...


# Translate between Python dictionaries and the strings Redis stores.
class RedisCache:
    def __init__(self, redis: RedisClient) -> None:
        self.redis = redis

    async def get(self, key: str) -> Optional[dict]:
        value = await self.redis.get(key)
        return json.loads(value) if value else None

    async def set(self, key: str, value: dict, ttl_seconds: int) -> None:
        await self.redis.set(
            key,
            json.dumps(value),
            ex=ttl_seconds,
        )

    async def delete(self, key: str) -> None:
        await self.redis.delete(key)


class FakeRedisClient:
    """Test adapter only; production uses redis.asyncio.Redis."""

    def __init__(self) -> None:
        self.values: dict[str, str] = {}

    async def get(self, key: str) -> Optional[str]:
        return self.values.get(key)

    async def set(self, key: str, value: str, *, ex: int) -> bool:
        self.values[key] = value
        return True

    async def delete(self, key: str) -> int:
        return int(self.values.pop(key, None) is not None)

    async def aclose(self) -> None:
        pass


# The composition root is the only place that constructs the real Redis client.
def build_production_cache(redis_url: str) -> tuple[RedisCache, RedisClient]:
    from redis.asyncio import Redis

    client = Redis.from_url(redis_url, decode_responses=True)
    return RedisCache(client), client


# Use the fake here so the notebook remains standalone and deterministic.
redis_client = FakeRedisClient()
cache: Cache = RedisCache(redis_client)

await cache.set("user:1", {"name": "Ada"}, ttl_seconds=300)
print(await cache.get("user:1"))

await cache.delete("user:1")
print(await cache.get("user:1"))

## Applied in this repository

The polished demo now uses the same JSON serialization, Redis `SET` with TTL, deletion, dependency injection, and async lifecycle used by [the LLM Redis adapter](../00P2-project-llm-api/app/infrastructure/cache.py). Production creates and closes the client in [main.py](../00P2-project-llm-api/app/main.py).